# Pipeline complet Deribit — BTC options de bout en bout

Ce notebook descend la pile de la plateforme sur des **vraies options BTC Deribit**
(API publique **testnet**, aucune authentification requise) en s'appuyant sur les
**use-cases d'orchestration de haut niveau** — pas de logique métier réimplémentée ici :

```
Connexion  →  Discovery  →  Capture (BrokerTick)  →  build_surface  →  Surface SVI  →  Risk  →  Scénarios
```

Discipline (cf. `notebooks/README.md`) : le notebook ne fait qu'**appeler et tracer** les engines
testés. Toute la calibration / le pricing / le risque vit dans
`algotrading.infra.orchestration` (`build_surface`, `run_incremental_analytics`) qui pilote l'acteur
unique. Le notebook ne contient ni formule ni seuil métier en dur — seulement le *glue* d'I/O
Deribit (REST → `BrokerTick`) et le tracé des sorties.

**Source : testnet `test.deribit.com`** (public). Run :
`uv run --group notebooks jupyter lab notebooks/demo_pipeline_deribit.ipynb`

In [ ]:
# %% Setup — imports, repo root, paramètres
from __future__ import annotations

from collections import defaultdict
from datetime import UTC, date, datetime
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np

# Config typée (versionnée → config_hash) — construite ici en glue, pas de seuil métier inventé.
from algotrading.core.config import (
    PlatformConfig,
    QcThresholdConfig,
    ScenarioConfig,
    SolverConfig,
    UniverseConfig,
    config_hash,
)

# Seam de collecte unifié (ADR 0027) : le push BrokerTick → l'acteur via les use-cases.
from algotrading.infra.collectors import BrokerTick, next_sequence
from algotrading.infra.connectivity import ManualClock
from algotrading.infra.contracts import InstrumentKey, InstrumentMaster, Position
from algotrading.infra.orchestration import (
    SurfaceJobRequest,
    build_surface,
    run_incremental_analytics,
)
from algotrading.infra.storage import ParquetStore

# Adaptateur Deribit (transport REST/WS + découverte d'instruments).
from algotrading.infra_deribit.collectors.deribit_discovery import discover_instruments
from algotrading.infra_deribit.connectivity.deribit_transport import DeribitTransport

# ── Cible & fenêtre de maturité ───────────────────────────────────────────────
CURRENCY = "BTC"  # BTC ou ETH
MIN_DTE = 7  # jours minimum jusqu'à expiry
MAX_DTE = 45  # jours maximum
STRIKE_BAND = 0.30  # ±30% autour du spot → un smile dense et fittable

# Deribit testnet : API publique, pas d'auth. Le mainnet serait www.deribit.com.
REST_BASE = "https://test.deribit.com/api/v2"
WS_BASE = "wss://test.deribit.com/ws/api/v2"

print(f"Cible : {CURRENCY} options, {MIN_DTE}–{MAX_DTE} DTE, strikes ±{STRIKE_BAND:.0%}")
print(f"Source : {REST_BASE}")

## 1 · Connexion Deribit & prix index

`DeribitTransport` est la couche réseau la plus basse (REST synchrone ici ; le WebSocket est
réservé à la capture live continue). On vérifie la connectivité testnet en lisant le prix index
BTC/USD via `/public/get_index_price`.

In [ ]:
# %% Connexion + prix index
INDEX_NAME = f"{CURRENCY.lower()}_usd"

transport = DeribitTransport(rest_base=REST_BASE, ws_base=WS_BASE)
index_data = transport.get("/public/get_index_price", {"index_name": INDEX_NAME})
INDEX_PRICE: float = float(index_data["index_price"])
print(f"✓ Connecté au testnet Deribit — {CURRENCY}/USD index : ${INDEX_PRICE:,.0f}")

## 2 · Découverte des instruments → InstrumentMaster

`discover_instruments` appelle `/public/get_instruments`, parse chaque nom Deribit
(ex. `BTC-27JUN26-100000-C`) vers un `OptionContract` canonique et filtre par fenêtre de maturité.
On en dérive les `InstrumentMaster` (l'identité économique + la clé canonique `InstrumentKey`)
que l'acteur consommera — c'est l'univers à souscrire et à valoriser.

In [ ]:
# %% Découverte instruments + masters
def _option_key(contract) -> InstrumentKey:
    """Clé canonique de l'acteur pour une option Deribit (glue : OptionContract → InstrumentKey)."""
    return InstrumentKey(
        underlying_symbol=contract.symbol,
        security_type="OPT",
        exchange="DERIBIT",
        currency="USD",
        multiplier=float(contract.multiplier),
        broker_contract_id=(
            f"{contract.symbol}-{contract.expiry}-{contract.strike}-{contract.right.value}"
        ),
        expiry=contract.expiry,
        strike=float(contract.strike),
        option_right=contract.right.value,
    )


# L'index BTC sert de sous-jacent de référence (un seul master, security_type CRYPTO).
underlying_key = InstrumentKey(
    underlying_symbol=CURRENCY,
    security_type="CRYPTO",
    exchange="DERIBIT",
    currency="USD",
    multiplier=1.0,
    broker_contract_id=CURRENCY,
)

AS_OF = datetime.now(UTC)
all_contracts = discover_instruments(transport, CURRENCY, min_days=MIN_DTE, max_days=MAX_DTE)

# Resserrer sur les strikes proches de la monnaie : un smile dense, pas la queue illiquide.
lo, hi = (1 - STRIKE_BAND) * INDEX_PRICE, (1 + STRIKE_BAND) * INDEX_PRICE
contracts = [c for c in all_contracts if lo <= float(c.strike) <= hi]

masters: list[InstrumentMaster] = [
    InstrumentMaster(
        instrument_key=underlying_key.canonical(),
        as_of_date=AS_OF.date(),
        instrument=underlying_key,
        raw_broker_payload="{}",
    )
]
contract_by_key: dict[str, object] = {}
for c in contracts:
    key = _option_key(c)
    contract_by_key[key.canonical()] = c
    masters.append(
        InstrumentMaster(
            instrument_key=key.canonical(),
            as_of_date=AS_OF.date(),
            instrument=key,
            raw_broker_payload="{}",
        )
    )

by_expiry_map: dict[date, int] = defaultdict(int)
for c in contracts:
    by_expiry_map[c.expiry] += 1
expiries_sorted = sorted(by_expiry_map)

print(f"Découverts : {len(all_contracts)}  →  retenus (±{STRIKE_BAND:.0%}) : {len(contracts)}")
print(f"Masters (1 sous-jacent + {len(contracts)} options) : {len(masters)}")
print(f"Expiries ({len(expiries_sorted)}) :")
for exp in expiries_sorted:
    dte = (exp - AS_OF.date()).days
    print(f"  {exp}  ({dte:3d} DTE)  —  {by_expiry_map[exp]} contrats")

In [ ]:
# %% Plot : distribution des contrats par DTE
dtes = [(exp - AS_OF.date()).days for exp in expiries_sorted]
counts = [by_expiry_map[exp] for exp in expiries_sorted]

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(dtes, counts, width=max(1, max(dtes) // 40), color="steelblue", alpha=0.85)
ax.set(xlabel="DTE (jours)", ylabel="Contrats", title=f"{CURRENCY} — Options par expiry (±band)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 3 · Capture REST → `BrokerTick` → `build_surface`

On lit un ticker REST par contrat (`/public/ticker`), on convertit les prix de la devise base
(BTC) en USD (`prix_btc × index`), et on émet des `BrokerTick` (le shape EAV push de la couche de
collecte unifiée, ADR 0027). Un petit **adaptateur push** rejoue ces ticks quand le collecteur le
*pompe* — exactement le contrat que `build_surface` attend (`adapter` + `drive`). C'est du *glue*
d'I/O ; aucune analytique ici.

> **Pourquoi REST et non WebSocket ?** Un notebook n'a pas de boucle `asyncio` permanente. Le
> WebSocket (`DeribitMarketDataAdapter`) est réservé à la capture continue. Pour un snapshot
> ponctuel, REST suffit et se rejoue cellule par cellule via l'adaptateur push.

In [ ]:
# %% Adaptateur push REST + fetch des tickers
class RestSnapshotAdapter:
    """MarketDataAdapter push minimal : rejoue une liste figée de BrokerTick quand pompé.

    C'est le même contrat que le collecteur unifié (set_tick_callback / subscribe / pump) ;
    ici la liste est constituée une fois par lecture REST testnet, pas par flux WebSocket.
    """

    def __init__(self, ticks: list[BrokerTick]) -> None:
        self._ticks = ticks
        self._cb = None

    def subscribe(self, instrument_keys: object) -> None: ...
    def set_tick_callback(self, callback) -> None:  # noqa: ANN001
        self._cb = callback
    def set_fault_callback(self, callback) -> None: ...  # noqa: ANN001
    def unsubscribe_all(self) -> None: ...

    def pump(self, _collector: object) -> None:
        for tick in self._ticks:
            self._cb(tick)  # type: ignore[misc]


def _deribit_name(contract) -> str:
    """Reconstruit le nom Deribit (jour SANS zéro de tête : BTC-6JUN26-60000-C)."""
    expiry_s = f"{contract.expiry.day}{contract.expiry.strftime('%b%y').upper()}"
    strike_s = str(int(contract.strike))
    return f"{contract.symbol}-{expiry_s}-{strike_s}-{contract.right.value}"


# Champs de prix Deribit (devise base sur le fil) → noms de champ EAV canoniques.
_DERIBIT_FIELDS = {
    "best_bid_price": "bid",
    "best_ask_price": "ask",
    "last_price": "last",
    "mark_price": "mark_price",
}

_counters: dict[tuple[str, str], int] = {}
ticks: list[BrokerTick] = []


def _emit(key: str, field: str, value: float) -> None:
    ticks.append(
        BrokerTick(
            instrument_key=key,
            field_name=field,
            value=value,
            underlying=CURRENCY,
            sequence=next_sequence(_counters, key, field),
            exchange_ts=AS_OF,
        )
    )


# Sous-jacent : un bid/ask serré autour de l'index + un last à l'index.
spread_half = INDEX_PRICE * 0.0001
_emit(underlying_key.canonical(), "bid", INDEX_PRICE - spread_half)
_emit(underlying_key.canonical(), "ask", INDEX_PRICE + spread_half)
_emit(underlying_key.canonical(), "last", INDEX_PRICE)

n_fetched, n_skipped = 0, 0
for canonical, contract in contract_by_key.items():
    try:
        ticker = transport.get("/public/ticker", {"instrument_name": _deribit_name(contract)})
    except Exception as exc:  # noqa: BLE001 — un contrat fautif ne doit pas avorter la boucle
        n_skipped += 1
        continue
    for src, canon in _DERIBIT_FIELDS.items():
        raw = ticker.get(src)
        if raw is not None and float(raw) > 0:
            _emit(canonical, canon, round(float(raw) * INDEX_PRICE, 6))  # BTC → USD
    n_fetched += 1

transport.close()
print(f"Tickers lus : {n_fetched}  |  ignorés : {n_skipped}")
print(f"BrokerTicks émis : {len(ticks)}")

In [ ]:
# %% build_surface — capture + acteur + surface SVI, en un seul use-case
# Config crypto : spreads BTC larges → seuils QC généreux. Versionnée → config_hash réel.
config = PlatformConfig(
    universe=UniverseConfig(version="deribit-btc-nb", underlyings=(CURRENCY,), exchange="DERIBIT"),
    qc_threshold=QcThresholdConfig(
        version="deribit-btc-nb",
        max_spread_pct=0.60,  # spreads BTC OTM larges
        max_quote_age_seconds=600.0,
        min_chain_count=2,
    ),
    solver=SolverConfig(version="deribit-btc-nb", iv_tolerance=1e-8, max_iterations=100),
    scenario=ScenarioConfig(
        version="deribit-btc-nb",
        spot_shocks=(-0.20, -0.10, 0.0, 0.10, 0.20),  # vol crypto >> actions
        vol_shocks=(-0.10, 0.0, 0.10),
    ),
)
CONFIG_HASH = config_hash(config)

adapter = RestSnapshotAdapter(ticks)

# Le store temporaire isole la run : build_surface (re)capture, dérive et persiste dedans.
_tmp = TemporaryDirectory(prefix="deribit-surface-")
store = ParquetStore(Path(_tmp.name))

surface = build_surface(
    request=SurfaceJobRequest(
        symbol=CURRENCY,
        trade_date=AS_OF.date(),
        market_data_type=3,  # 3 = delayed/last (testnet public)
        as_of=AS_OF,
        calc_ts=AS_OF,
        persist=True,
    ),
    store=store,
    config=config,
    config_hash=CONFIG_HASH,
    adapter=adapter,
    masters=masters,
    drive=adapter.pump,
    clock=ManualClock(start=AS_OF),
    correlation_id="notebook-deribit",
)

outputs = surface.outputs
print(f"Events capturés      : {surface.collection.event_count}")
print(f"Maturités fittées    : {surface.fitted_maturities}")
print(f"config_hash          : {CONFIG_HASH[:12]}...")
print(f"Snapshots / IvPoints : {len(outputs.snapshots)} / {len(outputs.iv_points)}")
print(f"Surface params (SVI) : {len(outputs.surface_parameters)}")

## 4 · Surface SVI calibrée

`build_surface` a piloté l'acteur : snapshot point-en-temps → QC → forward (parité put-call) →
inversion IV Black-76 → calibration SVI slice par slice. On lit les paramètres SVI persistés
(`outputs.surface_parameters`) et le résumé par maturité (`surface.slices`) — la sortie tracée
vers les seuils exacts via le `config_hash`.

In [ ]:
# %% Résumé des slices SVI
print(f"{'Expiry':>12} {'DTE':>4} {'méthode':>8} {'n':>4} {'ATM vol':>8} {'RMSE':>10} {'arb-free':>9}")
print("-" * 62)
for s in sorted(surface.slices, key=lambda s: s.maturity_years):
    dte = int(s.maturity_years * 365)
    rmse = f"{s.rmse:.5f}" if s.rmse is not None else "n/a"
    atm = f"{s.atm_vol * 100:.1f}%" if s.atm_vol is not None else "n/a"
    print(f"{str(s.expiry_date):>12} {dte:>4} {s.method:>8} {s.n_points:>4} {atm:>8} {rmse:>10} {str(s.arb_free):>9}")

# Provenance : la surface persistée trace son config_hash et sa version de modèle.
if outputs.surface_parameters:
    p0 = outputs.surface_parameters[0]
    print(f"\nmodel_version : {p0.model_version}")
    print(f"config_hash   : {p0.provenance.config_hash[:12]}...")

In [ ]:
# %% Plot : SVI fit vs points de marché, par expiry
from algotrading.infra.surfaces import SviParams

params_by_expiry = {p.expiry_date: p for p in outputs.surface_parameters}


def _expiry_of(contract_key: str) -> date | None:
    """L'expiry encodée dans la clé canonique (pipe-form) d'un IvPoint."""
    field = contract_key.split("|")[6]  # slot expiry de InstrumentKey.canonical()
    return date.fromisoformat(field) if field else None


# Points de marché bruts (k, variance totale) issus de l'inversion IV de l'acteur, par expiry.
market_by_expiry: dict[date, list[tuple[float, float]]] = defaultdict(list)
for iv in outputs.iv_points:
    if iv.k is None or iv.total_variance is None:
        continue
    exp = _expiry_of(iv.contract_key)
    if exp in params_by_expiry:
        market_by_expiry[exp].append((iv.k, iv.total_variance))

expiries_fit = sorted(params_by_expiry)
if not expiries_fit:
    print("Aucune slice — plot ignoré.")
else:
    fig, axes = plt.subplots(1, len(expiries_fit), figsize=(5 * len(expiries_fit), 4), squeeze=False)
    for idx, exp in enumerate(expiries_fit):
        ax = axes[0][idx]
        p = params_by_expiry[exp]
        svi = SviParams(a=p.svi_a, b=p.svi_b, rho=p.svi_rho, m=p.svi_m, sigma=p.svi_sigma)
        pts = market_by_expiry.get(exp, [])
        if pts:
            ks = [k for k, _ in pts]
            ax.scatter(ks, [v for _, v in pts], s=40, color="C1", label="Marché", zorder=5)
            k_grid = np.linspace(min(ks) - 0.05, max(ks) + 0.05, 200)
        else:
            k_grid = np.linspace(-0.4, 0.4, 200)
        ax.plot(k_grid, [svi.total_variance(float(k)) for k in k_grid], color="C0", lw=2, label="SVI")
        dte = int(p.maturity_years * 365)
        ax.set(xlabel="Log-moneyness ln(K/F)", ylabel="Variance totale σ²T", title=f"{exp} ({dte}j)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle(f"{CURRENCY} — Calibration SVI par expiry (sorties acteur)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# %% Plot : smile IV par expiry (depuis outputs.iv_points)
smiles: dict[date, list] = defaultdict(list)
for iv in outputs.iv_points:
    if iv.iv is None or iv.k is None:
        continue
    exp = _expiry_of(iv.contract_key)
    if exp in params_by_expiry:
        smiles[exp].append(iv)

if not smiles:
    print("Aucun IvPoint résolu — plot ignoré.")
else:
    exps = sorted(smiles)
    fig, axes = plt.subplots(1, len(exps), figsize=(5 * len(exps), 4), squeeze=False)
    for idx, exp in enumerate(exps):
        ax = axes[0][idx]
        pts = sorted(smiles[exp], key=lambda iv: iv.k)
        ax.plot([iv.k for iv in pts], [iv.iv * 100 for iv in pts], "o-", color="C0", ms=5)
        ax.axvline(0, color="grey", ls="--", lw=0.8, label="ATM")
        dte = (exp - AS_OF.date()).days
        ax.set(xlabel="Log-moneyness ln(K/F)", ylabel="IV (%)", title=f"{exp} ({dte}j)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle(f"{CURRENCY} — Smile de volatilité implicite (sorties acteur)", fontsize=13)
    plt.tight_layout()
    plt.show()

## 5 · Risque & scénarios — straddle ATM fictif

On valorise un **straddle ATM** (long 1 call + long 1 put au même strike) via
`run_incremental_analytics` sur le store déjà capturé : le même acteur calcule le risque agrégé
(Greeks nets) et rejoue chaque scénario du grid de la config (chocs spot/vol/temps). Toujours zéro
analytique dans le notebook — on injecte des positions et on lit les sorties.

In [ ]:
# %% Construire le straddle ATM + run_incremental_analytics
# On choisit le straddle dans une expiry RÉELLEMENT fittée (sinon pas de slice pour valoriser).
fitted_expiries = set(params_by_expiry)
options = [
    m for m in masters if m.instrument.is_option() and m.instrument.expiry in fitted_expiries
]
risk = None
scenarios: list = []

if not options:
    print("Aucune option sur une expiry fittée — risque ignoré.")
    call_leg = put_leg = None
else:
    atm = min(options, key=lambda m: abs(m.instrument.strike - INDEX_PRICE))
    atm_strike, atm_expiry = atm.instrument.strike, atm.instrument.expiry

    def _leg(right: str) -> InstrumentMaster | None:
        return next(
            (
                m
                for m in options
                if m.instrument.strike == atm_strike
                and m.instrument.expiry == atm_expiry
                and m.instrument.option_right == right
            ),
            None,
        )

    call_leg, put_leg = _leg("C"), _leg("P")

if (call_leg is None or put_leg is None) and options:
    print("Call ou put ATM introuvable au même strike/expiry — risque ignoré.")
elif call_leg is not None and put_leg is not None:
    positions = [
        Position(valuation_ts=AS_OF, portfolio_id="nb-straddle",
                 contract_key=call_leg.instrument.canonical(), quantity=1.0, source="notebook"),
        Position(valuation_ts=AS_OF, portfolio_id="nb-straddle",
                 contract_key=put_leg.instrument.canonical(), quantity=1.0, source="notebook"),
    ]
    analytics = run_incremental_analytics(
        store=store,
        config=config,
        config_hash=CONFIG_HASH,
        positions=positions,
        instruments=[m.instrument for m in masters],
        masters=masters,
        trade_date=AS_OF.date(),
        as_of=AS_OF,
        calc_ts=AS_OF,
        clock=ManualClock(start=AS_OF),
        correlation_id="notebook-deribit-risk",
        persist=False,
    )
    scenarios = list(analytics.outputs.scenarios)
    aggs = analytics.outputs.risk_aggregates
    risk = aggs[0] if aggs else None

    print(f"Straddle ATM : K={float(atm_strike):,.0f}  (call + put au même strike)")
    if risk is not None:
        print("\n--- Greeks nets du portefeuille (straddle ATM) ---")
        print(f"  Groupe : {risk.group_key}")
        print(f"  Delta  : {risk.net_delta:>+12.4f}  (≈0 attendu : straddle ATM)")
        print(f"  Gamma  : {risk.net_gamma:>+12.8f}")
        print(f"  Vega   : {risk.net_vega:>+12.2f}")
        print(f"  Theta  : {risk.net_theta:>+12.2f}")
    print(f"\nScénarios exécutés : {len({s.scenario_id for s in scenarios})}")

In [ ]:
# %% Plot : PnL par scénario (agrégé sur le portefeuille)
if not scenarios:
    print("Pas de scénarios — plot ignoré.")
else:
    pnl_by_scn: dict[str, float] = defaultdict(float)
    for s in scenarios:
        pnl_by_scn[s.scenario_id] += s.pnl
    names = list(pnl_by_scn)
    vals = [pnl_by_scn[n] for n in names]

    fig, ax = plt.subplots(figsize=(11, 5))
    colors = ["steelblue" if v >= 0 else "firebrick" for v in vals]
    ax.barh(names, vals, color=colors, alpha=0.85)
    ax.axvline(0, color="black", lw=0.8)
    ax.set(xlabel="PnL (USD)", title=f"{CURRENCY} straddle ATM — PnL par scénario (full reprice)")
    ax.grid(axis="x", alpha=0.3)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# Nettoyage du store temporaire.
_tmp.cleanup()

## Récapitulatif de la pile

| Étape | Module | Entrée → Sortie |
|-------|--------|-----------------|
| 1 Connexion | `infra_deribit.connectivity` | REST testnet → `index_price` |
| 2 Discovery | `infra_deribit.collectors.deribit_discovery` | REST → `OptionContract[]` → `InstrumentMaster[]` |
| 3 Capture | `infra.collectors` (`BrokerTick`) | tickers REST → `BrokerTick[]` (seam push, ADR 0027) |
| 4 Surface | `infra.orchestration.build_surface` | ticks + masters → acteur → `SurfaceJobResult` (slices SVI + outputs) |
| 5 Risk + scénarios | `infra.orchestration.run_incremental_analytics` | positions → `RiskAggregate[]` + `ScenarioResult[]` |

**Propriétés clés** : un seul driver (l'acteur) calibre et price ; le notebook n'appelle que les
use-cases d'orchestration testés et ne contient aucune formule. Provenance sur chaque dérivé
(`config_hash` imprimé en §3/§4). Source publique testnet, sans authentification.